In [ ]:
# 📌 1. Imports
import os
import cv2
import numpy as np
from math import log10
from tqdm import tqdm

# PSNR function
def calculate_psnr(img1, img2):
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:  # identical images
        return 100
    PIXEL_MAX = 255.0
    return 20 * log10(PIXEL_MAX / np.sqrt(mse))


In [3]:
# 📌 2. Function to compute PSNR per class
def compute_psnr_for_class(class_folder):
    real_images = [f for f in os.listdir(class_folder) if f.startswith("ISIC")]
    fake_images = [f for f in os.listdir(class_folder) if not f.startswith("ISIC")]
    
    psnr_scores = []
    
    for real, fake in tqdm(zip(real_images, fake_images), total=min(len(real_images), len(fake_images))):
        real_path = os.path.join(class_folder, real)
        fake_path = os.path.join(class_folder, fake)
        
        # load and resize to same size
        img1 = cv2.imread(real_path)
        img2 = cv2.imread(fake_path)
        
        if img1 is None or img2 is None:
            continue
        
        img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))
        
        psnr_val = calculate_psnr(img1, img2)
        psnr_scores.append(psnr_val)
    
    return np.mean(psnr_scores) if psnr_scores else None


In [5]:
# 📌 3. Compute PSNR for all classes
base_path = "C:\\Users\\devgo\\OneDrive\\Desktop\\fid\\data\\processed_images"
classes = os.listdir(base_path)

results = {}
all_scores = []

for cls in classes:
    cls_path = os.path.join(base_path, cls)
    if not os.path.isdir(cls_path):
        continue
    
    psnr_val = compute_psnr_for_class(cls_path)
    if psnr_val is not None:
        results[cls] = psnr_val
        all_scores.append(psnr_val)

# Overall average PSNR
results["Overall"] = np.mean(all_scores)

results


100%|██████████| 1113/1113 [00:00<00:00, 4390.60it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
100%|██████████| 142/142 [00:00<00:00, 4359.54it/s]


{'AKIEC': np.float64(28.123263541652825),
 'BCC': np.float64(28.12261570983672),
 'BKL': np.float64(28.041351102269875),
 'DF': np.float64(28.04313199341061),
 'MEL': np.float64(28.01911264212074),
 'VASC': np.float64(28.098025918412585),
 'Overall': np.float64(28.074583484617225)}